# Lesson 17 - Testing, Debugging, and Logging

**Goal:** Check meaningful boundaries and make expected failures observable without hiding defects.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. A function with a clear contract

Use small pure functions so checks can isolate behavior. assert is convenient for learning, but production validation must raise an exception because assertions can be disabled.

In [1]:
def annual_tax(annual_gross, threshold=1_200_000, rate=0.20):
    if annual_gross < 0:
        raise ValueError("Gross cannot be negative")
    return max(0, annual_gross - threshold) * rate

print(annual_tax(1_500_000))
assert annual_tax(1_200_000) == 0

60000.0


## 2. Boundary tests with unittest

Test below, exactly at, and above a threshold. Also test rejected input. subTest names each case. Load this notebook class explicitly instead of letting unittest inspect Jupyter command-line arguments.

In [2]:
import unittest
import io

class TaxTests(unittest.TestCase):
    def test_threshold(self):
        for gross, expected in [(1_199_999, 0), (1_200_000, 0), (1_200_100, 20)]:
            with self.subTest(gross=gross):
                self.assertAlmostEqual(annual_tax(gross), expected)

    def test_negative(self):
        with self.assertRaises(ValueError):
            annual_tax(-1)

test_output = io.StringIO()
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TaxTests)
test_result = unittest.TextTestRunner(stream=test_output, verbosity=2).run(suite)
print(test_output.getvalue())
assert test_result.wasSuccessful()

test_negative (builtins.TaxTests.test_negative) ... ok
test_threshold (builtins.TaxTests.test_threshold) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK



## 3. Log deliberately

Use a named logger and suitable levels: debug for detail, info for progress, warning for a recoverable issue, error for a failure. Avoid logging real personal or financial details unnecessarily. Configure a local handler and remove it afterward so rerunning does not duplicate messages.

In [3]:
import logging

log_stream = io.StringIO()
logger = logging.Logger("lesson17", level=logging.INFO)
handler = logging.StreamHandler(log_stream)
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)
try:
    logger.info("Processed %s sample records", 3)
    logger.warning("Rejected %s invalid record", 1)
finally:
    logger.removeHandler(handler)
    handler.close()
print(log_stream.getvalue())
assert "WARNING" in log_stream.getvalue()

INFO: Processed 3 sample records



## 4. Debug with a small reproducer

First reproduce a problem with minimal input, inspect intermediate values, then add a regression test. A numerical identity such as gross minus deductions equals net is useful alongside known-answer tests, not as the only test.

In [4]:
gross = 124_500
pf = 13_320
tax = 4_900
net = gross - pf - tax
assert net == 106_280
assert gross == net + pf + tax
print({"gross": gross, "pf": pf, "tax": tax, "net": net})

{'gross': 124500, 'pf': 13320, 'tax': 4900, 'net': 106280}


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. HRA boundaries

Test basic salaries just below 100,000, exactly 100,000, and above it.

**Check:** HRA never exceeds 16,000; the equality boundary is included correctly.

In [5]:
# Write your solution and checks here.

### Exercise 2. PF regression

Add a test that distinguishes PF on basic alone from PF on basic plus DA.

**Check:** Basic 60,000 must give PF 10,656, not 7,200.

In [6]:
# Write your solution and checks here.

### Exercise 3. Tax regression

Verify PF is not subtracted from annual gross before applying the tax threshold.

**Check:** Annual gross 1,500,000 produces annual tax 60,000 under the worksheet rules.

In [7]:
# Write your solution and checks here.

### Exercise 4. Parser tests

Test blank, negative, NaN, infinity, zero, and numeric text.

**Check:** Reject invalid cases and accept zero without catching unrelated programming errors.

In [8]:
# Write your solution and checks here.

### Exercise 5. Log counts

Process a small batch and log only accepted and rejected counts.

**Check:** One handler produces one copy of each message even on rerun.

In [9]:
# Write your solution and checks here.

## Chapter project

Create a unittest suite for your salary calculator covering known amounts, cap and threshold boundaries, invalid inputs, and Decimal rounding. Make the notebook fail visibly if the suite fails.

In [10]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `18_standard_library_patterns.ipynb` - Useful Standard Library Patterns.